In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response.content

'**Short answer:**  \nParrots talk because they are natural vocal learners. Their bodies, brains, and social habits are all wired to imitate the sounds they hear—whether the chirps of a mate, the alarm calls of a group, or—when kept as pets—the words of their human companions.\n\n---\n\n## 1. The biology that makes mimicry possible\n\n| Feature | Why it matters |\n|---------|----------------|\n| **Syrinx (the bird “voice box”)** | Parrots have an exceptionally flexible syrinx that can produce a wide range of sounds. It’s the key organ that lets them shape sounds in ways other birds can’t. |\n| **Large vocal learning brain** | The brain regions that control sound production (the *nucleus taeniomammillaris* and surrounding nuclei) are disproportionately large in parrots compared to other birds. This gives them the neural circuitry for learning and reproducing new sounds. |\n| **High vocal plasticity** | Even adult parrots can modify their calls. Their syrinx and brain can fine‑tune frequ

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"The weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather in Paris?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather.', 'tool_calls': [{'id': 'fc_be5121e6-1d1a-4ad3-aa8f-b336f3cb376a', 'function': {'arguments': '{"location":"Paris"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 126, 'total_tokens': 157, 'completion_time': 0.038016098, 'completion_tokens_details': {'reasoning_tokens': 8}, 'prompt_time': 0.006042067, 'prompt_tokens_details': None, 'queue_time': 0.00318544, 'total_time': 0.044058165}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0f7a4-7d89-7130-b269-f1f7894216ff-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Paris'}, 'id': 'fc_be5121e6-1d1a-4ad3-aa8f-b336f3cb376a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 126, 'output_tokens': 31, 'tota

### Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Paris?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Paris is sunny.
